# Bayesian Data Analysis (BDA3) in Google Colab with Google Drive & PyMC

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saurabh2086/saurabh2086.github.io/blob/main/bda3/notebooks/bda3_google_drive_setup_and_pymc.ipynb)

This notebook provides an automated, end-to-end environment for running Bayesian statistical models in Python using **Google Colab** and **Google Drive**.

### What this notebook does:
1. **Mounts Google Drive** for permanent storage of datasets, models, trace caches, and publication figures.
2. **Fetches & Standardizes Datasets** directly from Andrew Gelman's official BDA3 repository (`https://sites.stat.columbia.edu/gelman/book/data/`).
3. **Parses Archaic Formats** (`.asc`, `.dat`, `.csv`) into tidy, snake_case **CSV** and **Parquet** formats, preserving original raw files in a `raw/` archive.
4. **Builds a Reusable Loader (`bda3_loader`)** so any future notebook can load datasets with a single line: `df = load_bda3('rats')`.
5. **Implements 3 Canonical PyMC Models with Publication Figures**:
   - **Model 1 (Chapter 3)**: Speed of Light — Normal vs. Robust Student-$t$ for outlier resistance.
   - **Model 2 (Chapter 5)**: Rat Tumors — Hierarchical model with partial pooling / shrinkage visualization.
   - **Model 3 (Chapter 20)**: Golf Putting — Physics/geometry-based non-linear Bayesian model.

## 1. Mount Google Drive & Configure Workspace Directory

This mounts your Google Drive so that datasets, trace files, and generated figures persist across all your Colab sessions.

In [ ]:
import os
import sys

# Check if running inside Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print("==> Running inside Google Colab. Mounting Google Drive...")
    drive.mount('/content/drive')
    
    # Destination directory inside your Google Drive
    DATA_DIR = "/content/drive/MyDrive/BDA3_Data"
    FIGURES_DIR = "/content/drive/MyDrive/BDA3_Figures"
except ImportError:
    IN_COLAB = False
    print("==> Running locally outside Google Colab.")
    DATA_DIR = os.path.abspath("./bda3_data")
    FIGURES_DIR = os.path.abspath("./bda3_figures")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"Data Directory:    {DATA_DIR}")
print(f"Figures Directory: {FIGURES_DIR}")

## 2. Install / Verify Dependencies (PyMC, ArviZ, Matplotlib)

Google Colab includes PyMC pre-installed or easily updated via pip.

In [ ]:
!pip install -q "pymc>=5.10.0" "arviz>=0.17.0" "pyarrow>=14.0.0" watermark

import pymc as pm
import arviz as az
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set publication-ready plot style
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 120
})

%load_ext watermark
%watermark -p pymc,arviz,pandas,numpy,matplotlib

## 3. Automated ETL Pipeline: Download & Format BDA3 Datasets

This cell downloads the raw files from Columbia University, parses comments and custom spacing, and exports both `.csv` and `.parquet` into your Google Drive.

In [ ]:
import re
import urllib.request

BASE_URL_SITES = "https://sites.stat.columbia.edu/gelman/book/data/"
BASE_URL_STAT = "https://stat.columbia.edu/~gelman/book/data/"

raw_dir = os.path.join(DATA_DIR, "raw")
csv_dir = os.path.join(DATA_DIR, "clean_csv")
parquet_dir = os.path.join(DATA_DIR, "parquet")

os.makedirs(raw_dir, exist_ok=True)
os.makedirs(csv_dir, exist_ok=True)
os.makedirs(parquet_dir, exist_ok=True)

def fetch_text(url: str) -> str:
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (BDA3-Pipeline)"})
    with urllib.request.urlopen(req) as resp:
        return resp.read().decode("utf-8", errors="replace")

# Parsers
def parse_rats(text):
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    data = []
    found = False
    for line in lines:
        if line.startswith("y N") or line.startswith("y\tN"):
            found = True; continue
        if found:
            p = line.split()
            if len(p) >= 2:
                data.append((int(p[0]), int(p[1])))
    df = pd.DataFrame(data, columns=["y", "n"])
    df.insert(0, "study_id", np.arange(1, len(df) + 1))
    df["observed_rate"] = (df["y"] / df["n"]).round(4)
    return df

def parse_light(text):
    lines = text.strip().splitlines()
    tokens = []
    for line in lines:
        pts = line.strip().split()
        if pts and all(re.match(r"^-?\d+$", p) for p in pts):
            tokens.extend([int(p) for p in pts])
    df = pd.DataFrame({"obs_id": np.arange(1, len(tokens) + 1), "deviation_ns": tokens})
    df["passage_time_ns"] = 24800 + df["deviation_ns"]
    df["speed_km_s"] = (7.442 / (df["passage_time_ns"] * 1e-9)).round(2)
    return df

def parse_golf(text):
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    rows = []
    for line in lines[1:]:
        p = line.split()
        if len(p) >= 3:
            rows.append([int(p[0]), int(p[1]), int(p[2])])
    df = pd.DataFrame(rows, columns=["distance_ft", "n_attempts", "y_success"])
    df["success_rate"] = (df["y_success"] / df["n_attempts"]).round(4)
    return df

def parse_meta(text):
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    rows = []
    found = False
    for line in lines:
        if "study" in line and "control" in line: found = True; continue
        if found:
            p = line.split()
            if len(p) >= 5: rows.append([int(x) for x in p[:5]])
    df = pd.DataFrame(rows, columns=["study_id", "control_deaths", "control_total", "treated_deaths", "treated_total"])
    df["control_rate"] = (df["control_deaths"] / df["control_total"]).round(4)
    df["treated_rate"] = (df["treated_deaths"] / df["treated_total"]).round(4)
    return df

def parse_fabric(text):
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    rows = []
    found = False
    for line in lines:
        if "length" in line and "faults" in line: found = True; continue
        if found:
            p = line.split()
            if len(p) >= 2: rows.append([int(p[0]), int(p[1])])
    df = pd.DataFrame(rows, columns=["length", "faults"])
    df.insert(0, "roll_id", np.arange(1, len(df) + 1))
    df["faults_per_100m"] = ((df["faults"] / df["length"]) * 100).round(3)
    return df

def parse_schiz(text):
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    matrix = []
    for line in lines:
        pts = line.split()
        if len(pts) >= 10 and all(p.isdigit() for p in pts):
            matrix.append([int(p) for p in pts])
    records = []
    for idx, row in enumerate(matrix):
        subj_id = idx + 1
        is_sc = 1 if idx >= 11 else 0
        grp = "schizophrenic" if is_sc == 1 else "control"
        for t_idx, val in enumerate(row, 1):
            records.append({"subject_id": subj_id, "group": grp, "is_schizophrenic": is_sc, "trial": t_idx, "reaction_time_ms": val, "log_rt": round(np.log(val), 4)})
    return pd.DataFrame(records)

def parse_football(text):
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    rows = []
    found = False
    for line in lines:
        if "favorite" in line and "spread" in line: found = True; continue
        if found:
            p = line.split()
            if len(p) >= 7:
                try:
                    rows.append({"home_favored": int(p[0]), "favorite_score": int(p[1]), "underdog_score": int(p[2]), "spread": float(p[3]), "favorite_team": p[4], "underdog_team": p[5], "week": int(p[6])})
                except ValueError:
                    pass
    df = pd.DataFrame(rows)
    df.insert(0, "game_id", np.arange(1, len(df) + 1))
    df["score_diff"] = df["favorite_score"] - df["underdog_score"]
    return df

pipeline = [
    ("rats", BASE_URL_SITES + "rats.asc", "rats.asc", parse_rats),
    ("light", BASE_URL_SITES + "light.asc", "light.asc", parse_light),
    ("golf", BASE_URL_SITES + "golf.dat", "golf.dat", parse_golf),
    ("meta", BASE_URL_SITES + "meta.asc", "meta.asc", parse_meta),
    ("fabric", BASE_URL_SITES + "fabric.asc", "fabric.asc", parse_fabric),
    ("schizophrenia", BASE_URL_SITES + "schiz.asc", "schiz.asc", parse_schiz),
    ("football", BASE_URL_SITES + "football.asc", "football.asc", parse_football),
]

print(f"Starting download & parse to {DATA_DIR}...")
for name, url, raw_name, parser in pipeline:
    try:
        raw_text = fetch_text(url)
        with open(os.path.join(raw_dir, raw_name), "w", encoding="utf-8") as f:
            f.write(raw_text)
        df = parser(raw_text)
        df.to_csv(os.path.join(csv_dir, f"{name}.csv"), index=False)
        df.to_parquet(os.path.join(parquet_dir, f"{name}.parquet"), index=False)
        print(f"  [OK] {name:15} -> {len(df):4d} rows, cols: {list(df.columns)}")
    except Exception as e:
        print(f"  [ERR] {name}: {e}")

print("\nAll datasets processed and saved to Google Drive!")

## 4. Reusable Loader Function

Use this helper to quickly load any cleaned dataset as a Pandas DataFrame.

In [ ]:
def load_bda3(name: str) -> pd.DataFrame:
    """Load a BDA3 dataset from Parquet (preferred) or CSV."""
    p_path = os.path.join(DATA_DIR, "parquet", f"{name}.parquet")
    c_path = os.path.join(DATA_DIR, "clean_csv", f"{name}.csv")
    if os.path.exists(p_path):
        return pd.read_parquet(p_path)
    elif os.path.exists(c_path):
        return pd.read_csv(c_path)
    raise FileNotFoundError(f"Dataset {name} not found in {DATA_DIR}")

# Quick sanity check
df_rats = load_bda3('rats')
print("Rat Tumor Dataset (Table 5.1):")
df_rats.head()

## 5. Model 1 (Chapter 3): Simon Newcomb's Speed of Light
### Comparing Normal Likelihood vs. Robust Student-$t$

Newcomb measured the passage time of light over a 7.442 km path. Two of the 66 measurements are clear negative outliers ($-44$ and $-2$). A standard Normal likelihood gets dragged down by these outliers. A Student-$t$ likelihood with inferable degrees of freedom $\nu$ downweights the outliers automatically.

In [ ]:
df_light = load_bda3('light')
y_light = df_light['deviation_ns'].values

print(f"Measurements: n = {len(y_light)}")
print(f"Summary: mean = {y_light.mean():.2f}, std = {y_light.std(ddof=1):.2f}, min = {y_light.min()}, max = {y_light.max()}")

# Fit Student-t Robust Bayesian Model
with pm.Model() as model_light_robust:
    # Priors
    mu = pm.Normal("mu", mu=25, sigma=15)
    sigma = pm.HalfNormal("sigma", sigma=10)
    nu = pm.Exponential("nu", lam=1/29.0) + 1.0  # shifted exponential for degrees of freedom
    
    # Likelihood
    likelihood = pm.StudentT("y_obs", nu=nu, mu=mu, sigma=sigma, observed=y_light)
    
    # Sample MCMC
    idata_light = pm.sample(draws=1500, tune=1000, chains=4, random_seed=42, return_inferencedata=True)
    
    # Posterior Predictive
    ppc_light = pm.sample_posterior_predictive(idata_light, random_seed=42)

az.summary(idata_light, var_names=["mu", "sigma", "nu"])

In [ ]:
# Publication Figure: Speed of Light Posterior & Data Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=300)

# Panel A: Data & PPC
axes[0].hist(y_light, bins=25, density=True, color="#4C72B0", alpha=0.6, edgecolor="white", label="Observed Data")
ppc_samples = ppc_light.posterior_predictive["y_obs"].values.flatten()
sns.kdeplot(ppc_samples, ax=axes[0], color="#C44E52", lw=2, label="Student-t Posterior Predictive")
axes[0].axvline(y_light.mean(), color="black", ls="--", lw=1.2, label=f"Sample Mean ({y_light.mean():.1f})")
axes[0].set_title("(A) Simon Newcomb Speed of Light & PPC", fontweight="bold")
axes[0].set_xlabel("Deviation from 24,800 ns")
axes[0].set_ylabel("Density")
axes[0].set_xlim(-50, 50)
axes[0].legend(frameon=True)

# Panel B: Posterior Distribution of Mu
az.plot_posterior(idata_light, var_names=["mu"], ax=axes[1], hdi_prob=0.94, color="#55A868")
axes[1].set_title("(B) Posterior for True Location $\\mu$ (Robust)", fontweight="bold")
axes[1].set_xlabel("\\mu (Deviation in ns)")

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "ch03_speed_of_light_robust_model.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Saved publication figure to: {fig_path}")
plt.show()

## 6. Model 2 (Chapter 5): Hierarchical Rat Tumor Rates
### Partial Pooling and Shrinkage Estimation Across 71 Studies

In 71 historical control experiments, the sample size per group varies from 10 to 52. Estimating each group's tumor rate independently leads to noisy estimates in small groups. Full pooling ignores group differences. A hierarchical Beta-Binomial / Logit-Normal model learns the population distribution and shrinks noisy observations toward the grand mean.

In [ ]:
df_rats = load_bda3('rats')
y_rats = df_rats['y'].values
n_rats = df_rats['n'].values
K_groups = len(y_rats)

with pm.Model() as model_rats_hierarchical:
    # Hyper-priors on population log-odds
    mu_pop = pm.Normal("mu_pop", mu=-2.0, sigma=1.0)        # mean log-odds ~ 12% tumor rate
    sigma_pop = pm.HalfNormal("sigma_pop", sigma=0.8)       # across-study standard deviation
    
    # Non-centered parameterization for group-level log-odds
    z_group = pm.Normal("z_group", mu=0, sigma=1, shape=K_groups)
    theta_logit = pm.Deterministic("theta_logit", mu_pop + z_group * sigma_pop)
    theta = pm.Deterministic("theta", pm.math.invlogit(theta_logit))
    
    # Binomial Likelihood
    y_obs = pm.Binomial("y_obs", n=n_rats, p=theta, observed=y_rats)
    
    # MCMC Sampling
    idata_rats = pm.sample(draws=1500, tune=1000, chains=4, target_accept=0.95, random_seed=42, return_inferencedata=True)

az.summary(idata_rats, var_names=["mu_pop", "sigma_pop"])

In [ ]:
# Publication Figure: Shrinkage / Partial Pooling Visualization
theta_posterior_mean = idata_rats.posterior["theta"].mean(dim=["chain", "draw"]).values
hdi_theta = az.hdi(idata_rats.posterior["theta"], hdi_prob=0.94)["theta"].values

pop_mean_rate = np.exp(idata_rats.posterior["mu_pop"].mean().values) / (1 + np.exp(idata_rats.posterior["mu_pop"].mean().values))

fig, ax = plt.subplots(figsize=(11, 6), dpi=300)

raw_rates = df_rats["observed_rate"].values
study_indices = np.arange(1, K_groups + 1)

# Sort by raw observed rate for clear visualization
sort_idx = np.argsort(raw_rates)

ax.scatter(np.arange(K_groups), raw_rates[sort_idx], color="#C44E52", s=35, alpha=0.8, label="Raw Observed Rate ($y_j / n_j$)", zorder=3)
ax.scatter(np.arange(K_groups), theta_posterior_mean[sort_idx], color="#4C72B0", s=35, label="Hierarchical Bayes Mean $\\hat{\\theta}_j$", zorder=4)

# Connect raw and shrunk estimates
for i in range(K_groups):
    ax.vlines(i, ymin=min(raw_rates[sort_idx][i], theta_posterior_mean[sort_idx][i]),
              ymax=max(raw_rates[sort_idx][i], theta_posterior_mean[sort_idx][i]),
              color="gray", alpha=0.5, lw=1)

ax.axhline(pop_mean_rate, color="black", ls="--", lw=1.5, label=f"Population Grand Mean ({pop_mean_rate:.1%})")

ax.set_title("Shrinkage / Partial Pooling in the BDA3 Rat Tumor Studies (Table 5.1)", fontweight="bold")
ax.set_xlabel("Study (Sorted by Raw Observed Rate)")
ax.set_ylabel("Tumor Probability $\\theta_j$")
ax.set_ylim(-0.02, 0.45)
ax.legend(frameon=True, loc="upper left")

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "ch05_rat_tumors_shrinkage_partial_pooling.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Saved publication figure to: {fig_path}")
plt.show()

## 7. Model 3 (Chapter 20): Golf Putting Success
### Geometry / Physics-Based Non-Linear Model

In Chapter 20, Gelman introduces the famous golf putting model derived from physical geometry: a putt is successful if the initial angle error $\theta$ is small enough that the ball enters the hole:

$$|\theta| < \arcsin\left( \frac{R - r}{x} \right)$$

where $R = 4.25/2$ inches (radius of hole), $r = 1.68/2$ inches (radius of golf ball), and $x$ is the distance in feet (converted to inches). Assuming angle errors are $\mathcal{N}(0, \sigma^2)$, the probability of making the putt is:

$$P(\text{success} \mid x) = 2 \Phi\left( \frac{\arcsin\left( \frac{R - r}{x} \right)}{\sigma} \right) - 1$$

Let's estimate the single dispersion parameter $\sigma$ using PyMC!

In [ ]:
import pytensor.tensor as pt

df_golf = load_bda3('golf')

x_feet = df_golf["distance_ft"].values
x_inches = x_feet * 12.0
n_trials = df_golf["n_attempts"].values
y_trials = df_golf["y_success"].values

# Ball and hole dimensions (inches)
R_hole = 4.25 / 2.0
r_ball = 1.68 / 2.0
threshold_angle = np.arcsin((R_hole - r_ball) / x_inches)

with pm.Model() as model_golf_physics:
    # Prior on putting angle standard deviation (in radians)
    # Gelman uses prior centered around ~ 0.02 radians (~ 1.1 degrees)
    sigma_angle = pm.HalfNormal("sigma_angle", sigma=0.05)
    
    # Physics success probability
    z = threshold_angle / sigma_angle
    # Standard normal CDF = 0.5 * (1 + erf(z / sqrt(2)))
    phi_z = 0.5 * (1.0 + pt.erf(z / np.sqrt(2.0)))
    p_success = pm.Deterministic("p_success", 2.0 * phi_z - 1.0)
    
    # Binomial likelihood
    y_obs = pm.Binomial("y_obs", n=n_trials, p=p_success, observed=y_trials)
    
    idata_golf = pm.sample(draws=1500, tune=1000, chains=4, random_seed=42, return_inferencedata=True)

az.summary(idata_golf, var_names=["sigma_angle"])

In [ ]:
# Publication Figure: Golf Putting Fit with 94% Credible Band
sigma_samples = idata_golf.posterior["sigma_angle"].values.flatten()
sigma_deg = np.rad2deg(sigma_samples.mean())

x_grid_feet = np.linspace(2, 21, 200)
x_grid_inches = x_grid_feet * 12.0
threshold_grid = np.arcsin((R_hole - r_ball) / x_grid_inches)

# Compute curve for each posterior sample
from scipy.special import erf
curve_samples = np.array([
    2.0 * (0.5 * (1.0 + erf((threshold_grid / s) / np.sqrt(2.0)))) - 1.0
    for s in sigma_samples[::5]  # subsample for speed
])

curve_mean = curve_samples.mean(axis=0)
curve_hdi_low = np.percentile(curve_samples, 3, axis=0)
curve_hdi_high = np.percentile(curve_samples, 97, axis=0)

fig, ax = plt.subplots(figsize=(9, 5), dpi=300)

# Observed Data Points
ax.scatter(df_golf["distance_ft"], df_golf["success_rate"], color="#C44E52", s=60, zorder=5,
           edgecolor="black", lw=0.7, label="Observed Data (1988 PGA Tour)")

# Posterior Mean Curve and 94% HDI Band
ax.plot(x_grid_feet, curve_mean, color="#4C72B0", lw=2.5,
        label=f"Geometry Model Mean ($\\sigma \\approx {sigma_deg:.2f}^\\circ$)")
ax.fill_between(x_grid_feet, curve_hdi_low, curve_hdi_high, color="#4C72B0", alpha=0.25,
                label="94% Credible Interval")

ax.set_title("Gelman's Physical-Geometry Golf Putting Model (BDA3 Table 20.1)", fontweight="bold")
ax.set_xlabel("Distance to Hole (Feet)")
ax.set_ylabel("Probability of Success")
ax.set_ylim(-0.02, 1.05)
ax.set_xlim(1.5, 21.5)
ax.legend(frameon=True, loc="upper right")

plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "ch20_golf_putting_geometry_model.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Saved publication figure to: {fig_path}")
plt.show()

## 8. Summary of Saved Assets in Google Drive

Run this final cell to confirm the contents of your Google Drive directories.

In [ ]:
print("=== Datasets in Google Drive (clean_csv/) ===")
for f in sorted(os.listdir(csv_dir)):
    size_kb = os.path.getsize(os.path.join(csv_dir, f)) / 1024
    print(f"  - {f:25} ({size_kb:6.1f} KB)")

print("\n=== Figures in Google Drive (BDA3_Figures/) ===")
for f in sorted(os.listdir(FIGURES_DIR)):
    size_kb = os.path.getsize(os.path.join(FIGURES_DIR, f)) / 1024
    print(f"  - {f:35} ({size_kb:6.1f} KB)")

print("\nSetup complete! You can now use these clean datasets and figures in your articles.")